In [ ]:
%load_ext autoreload
%autoreload 2

import os
os.environ['HF_HOME'] = '/tmp/wendler/.hfcache'

In [ ]:
import torch
import sys
sys.path.append('../')
from SDLens import HookedStableDiffusionXLPipeline

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

dtype = torch.float32
    
pipe_inference = HookedStableDiffusionXLPipeline.from_pretrained("stabilityai/sdxl-turbo", 
                                                                 torch_dtype=dtype,
                                                                 device_map="balanced",
                                                                 variant=("fp16" if dtype==torch.float16 else None)
                                                                )

In [ ]:
if dtype == torch.float32:
    pipe_inference.text_encoder_2.to(dtype)

In [ ]:
import re
resnet_blocks = set()
conv_shortcut_set = set()
for n in pipe_inference.unet.named_modules():
    m = re.match(r"(.+resnets\.\d+)$", n[0])
    if m:
        block_name = m.group(1)
        resnet_blocks.add(block_name)
        for n2 in pipe_inference.unet.named_modules():
            if "conv_shortcut" in n2[0]:
                if block_name in n2[0]:
                    conv_shortcut_set.add(block_name.replace(".conv_shortcut",""))
print(resnet_blocks)
resnet_blocks_list = sorted(list(resnet_blocks))
print(resnet_blocks_list)

In [ ]:
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

prompts = [
    "a photo of a colorful model",
    "a painting of a futuristic city at sunset",
    "a close-up of a cat wearing sunglasses",
    "a landscape of snowy mountains under a starry sky",
    "a futuristic robot in a neon-lit alley",
    "a serene beach at sunrise with palm trees",
    "a portrait of a woman with rainbow hair",
    "a bustling market in a medieval town",
    "a fantasy castle floating in the clouds",
    "a macro shot of a dew-covered spider web",
    "a cyberpunk street scene at night",
    "a magical forest with glowing mushrooms",
    "a vintage car parked in front of a diner",
    "a group of astronauts on an alien planet",
    "a dragon flying over a burning village",
    "a steampunk airship in the sky",
    "a child playing with a puppy in a field",
    "a surreal landscape with melting clocks",
    "a majestic lion resting in tall grass",
    "a futuristic train speeding through a desert"
]

In [ ]:
from functools import partial
from utils import TimeDependentHook

import open_clip
from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity
import PIL
import torch

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm

# CLIP scorer class
class CLIPScorer:
    def __init__(self, model_name='ViT-L-14'):
        self.model_name = model_name
        self.model, _, self.preprocess = open_clip.create_model_and_transforms(model_name, pretrained='openai')
        self.model.cuda()
        self.model.eval()
        self.tokenizer = open_clip.get_tokenizer(model_name)

    def embed_texts(self, texts):
        with torch.no_grad(), torch.cuda.amp.autocast():
            text = self.tokenizer(texts).cuda()
            text_features = self.model.encode_text(text)
        return text_features

    def embed_images(self, images):
        with torch.no_grad(), torch.cuda.amp.autocast():
            tensors = []
            for img in images:
                if isinstance(img, np.ndarray):
                    img = Image.fromarray(img)
                tensor = self.preprocess(img).unsqueeze(0)
                tensors += [tensor]
            tensors = torch.cat(tensors, dim=0)
            image_features = self.model.encode_image(tensors.cuda())
        return image_features

    def get_scores(self, texts, images, normalize=True):
        text_features = self.embed_texts(texts)
        image_features = self.embed_images(images)
        if normalize:
            text_features /= text_features.norm(dim=-1, keepdim=True)
            image_features /= image_features.norm(dim=-1, keepdim=True)
        scores = (text_features @ image_features.T)
        return scores

    def get_scores_images(self, images1, images2, normalize=True):
        image_features1 = self.embed_images(images1)
        image_features2 = self.embed_images(images2)
        if normalize:
            image_features1 /= image_features1.norm(dim=-1, keepdim=True)
            image_features2 /= image_features2.norm(dim=-1, keepdim=True)
        scores = (image_features1 @ image_features2.T)
        return scores

def img2lpips(img):
    if isinstance(img, PIL.Image.Image):
        img = np.array(img)
        return (torch.tensor(img).float()/255.).unsqueeze(0).permute(0, 3, 1, 2)
    if isinstance(img, np.ndarray) and img.dtype == np.uint8:
        return (torch.tensor(img).float()/255.).unsqueeze(0).permute(0, 3, 1, 2)
    else:
        return img

lpips_metric = LearnedPerceptualImagePatchSimilarity(net_type='alex', normalize=True).cuda()
clip_scorer = CLIPScorer()

all_results = []

def ablate_resnet(module, inputs, outputs, idx, surrogate_output=None):
    if surrogate_output is not None:
        return surrogate_output[:, idx]
    return inputs[0]

for prompt in prompts: 
    conv_shortcut_list = sorted(list(conv_shortcut_set))
    conv_shortcut_list = [f"unet.{name}.conv_shortcut" for name in conv_shortcut_list]
    ref, cache = pipe_inference.run_with_cache(
                    prompt,
                    positions_to_cache=conv_shortcut_list,
                    num_inference_steps=1,
                    generator=torch.Generator(device=device).manual_seed(42),
                    resolution=512,
                    guidance_scale=0.0,
                )
    print(cache['output']['unet.down_blocks.1.resnets.0.conv_shortcut'].shape)

    images = []
    block_names_for_grid = []

    for block_name in resnet_blocks_list:
        try: 
            if block_name in conv_shortcut_set:
                hook_fn = partial(ablate_resnet, surrogate_output=cache['output'][f"unet.{block_name}.conv_shortcut"])
                hook = TimeDependentHook(hook_fn, 1, apply_at_steps=[0])#, 1, 2, 3])
            else:
                hook = partial(ablate_resnet, idx=0)
            print(f"Ablating {block_name}")
            full_block_name = f'unet.{block_name}'

            out = pipe_inference.run_with_hooks(
                prompt,
                position_hook_dict={full_block_name: hook},
                num_inference_steps=1,
                generator=torch.Generator(device=device).manual_seed(42),
                guidance_scale=0.0,
            )
            images.append(out.images[0])
            block_names_for_grid.append(block_name)
        except Exception as e:
            print(f"Failed to ablate {block_name}: {e}")

    # Convert PIL images to tensors for make_grid
    def pil_to_tensor(img):
        img_np = np.array(img)
        print(img_np.min(), img_np.max(), img_np.mean())
        return torch.from_numpy(np.array(img)).permute(2, 0, 1)

    # Add the reference image as the first image in the grid
    image_tensors = [pil_to_tensor(ref.images[0])] + [pil_to_tensor(img) for img in images]
    labels = ["Reference"] + block_names_for_grid

    n_images = len(image_tensors)
    nrow = 4  # Adjust as needed
    ncol = (n_images + nrow - 1) // nrow

    # Plot with block names as titles
    plt.figure(figsize=(4 * nrow, 4 * ncol))
    for idx, (img_tensor, label) in enumerate(zip(image_tensors, labels)):
        plt.subplot(ncol, nrow, idx + 1)
        img = img_tensor.permute(1, 2, 0).cpu().numpy().astype(np.uint8)
        plt.imshow(img)
        plt.title(label, fontsize=10)
        plt.axis('off')
    plt.tight_layout()
    plt.savefig(f"../results/resnet_ablation/{prompt}.png")
    plt.close()

    # --- Compute LPIPS and CLIP scores ---
    # LPIPS: compare each ablated image to the reference
    ref_lpips_img = img2lpips(ref.images[0]).cuda()
    ablated_lpips_imgs = [img2lpips(img).cuda() for img in images]
    lpips_scores = []
    for ablated_img in ablated_lpips_imgs:
        with torch.no_grad():
            score = lpips_metric(ref_lpips_img, ablated_img).item()
        lpips_scores.append(score)

    # CLIP: compare each ablated image to the prompt
    # (Reference image also included for comparison)
    all_images_for_clip = [ref.images[0]] + images
    with torch.no_grad():
        clip_scores = clip_scorer.get_scores([prompt], all_images_for_clip).cpu().numpy().flatten().tolist()

    # Save scores for this prompt to all_results
    for i, block in enumerate(["Reference"] + block_names_for_grid):
        all_results.append({
            "prompt": prompt,
            "block": block,
            "lpips_to_ref": 0.0 if i == 0 else lpips_scores[i-1],
            "clip_score": clip_scores[i]
        })

# After all prompts, aggregate into a DataFrame
results_df = pd.DataFrame(all_results)
results_df.to_csv(f"../results/resnet_ablation/all_prompts_scores.csv", index=False)
print(results_df)

# Compute mean, std, and gaussian 95% confidence intervals for each block over all prompts
def gaussian_95ci(std, n):
    # 95% CI for mean: mean ± 1.96 * std / sqrt(n)
    return 1.96 * std / np.sqrt(n)

agg = results_df.groupby("block").agg(
    lpips_mean=("lpips_to_ref", "mean"),
    lpips_std=("lpips_to_ref", "std"),
    clip_mean=("clip_score", "mean"),
    clip_std=("clip_score", "std"),
    count=("prompt", "count")
).reset_index()

agg["lpips_95ci"] = gaussian_95ci(agg["lpips_std"], agg["count"])
agg["clip_95ci"] = gaussian_95ci(agg["clip_std"], agg["count"])

print(agg)

# Serialize the table with mean, std, and gaussian 95% CI
agg.to_csv(f"../results/resnet_ablation/ablation_scores_summary_stats.csv", index=False)

# Plotting with gaussian 95% confidence intervals
fig, axs = plt.subplots(1, 2, figsize=(14, 5))

# LPIPS plot
axs[0].errorbar(
    agg["block"], agg["lpips_mean"], yerr=agg["lpips_95ci"], fmt='o', capsize=3
)
axs[0].set_title("LPIPS to Reference (mean ± 95% CI over prompts)")
axs[0].set_ylabel("LPIPS")
axs[0].set_xlabel("Block")
axs[0].tick_params(axis='x', rotation=90)

# CLIP plot
axs[1].errorbar(
    agg["block"], agg["clip_mean"], yerr=agg["clip_95ci"], fmt='o', capsize=3
)
axs[1].set_title("CLIP Score wrt Prompt (mean ± 95% CI over prompts)")
axs[1].set_ylabel("CLIP Score")
axs[1].set_xlabel("Block")
axs[1].tick_params(axis='x', rotation=90)

plt.tight_layout()
plt.savefig(f"../results/resnet_ablation/ablation_scores_summary.png")
plt.close()